In [1]:
from datasets import load_dataset

ds = load_dataset("AIGuruTinix/ViBioMIR", 'corpus')

README.md: 0.00B [00:00, ?B/s]

links_corpus.parquet:   0%|          | 0.00/35.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [2]:
print(ds["train"][0])

{'id': 583, 'url': 'http://special.vietnamplus.vn/benh_vien_nhan_ai'}


In [3]:
! pip install playwright
! playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 30.0 MB/s eta 0:00:00:00:0100:01
186.8 MiB [                    ] 0% 384.7s186.8 MiB [                    ] 0% 351.1s186.8 MiB [                    ] 0% 425.8s186.8 MiB [                    ] 0% 260.3s186.8 MiB [                    ] 0% 81.9s186.8 MiB [                    ] 0% 36.9s186.8 MiB [                    ] 1% 19.4s186.8 MiB [                    ] 1% 16.7s186.8 MiB [                    ] 2% 14.6s186.8 MiB [                    ] 2% 13.4s186.8 MiB [=                   ] 2% 12.5s186.8 MiB [=                   ] 3% 11.5s186.8 MiB [=                   ] 3% 10.2s186.8 MiB [=                   ] 3% 9.5s186.8 MiB [=                   ] 4% 9.0s186.8 MiB [=                   ] 4% 8.6s186.8 MiB [=                   ] 4% 8.5s186.8 MiB [=                   ] 5% 7.9s186.8 MiB [=                   ] 5% 7.7s186.8 MiB [=                   ] 6% 7.5s186.8 MiB [=                   ] 6% 7.6s186.8 MiB [=                   ] 6% 7.2s186.8 MiB [= 

In [4]:
!playwright install-deps chromium

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [114 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,321 kB]
Get:13 http://security.ubunt

In [5]:
!playwright install chromium

In [7]:
! pip install --upgrade trafilatura

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.6/257.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 18.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 38.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 8.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 7.8 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: lxml
    Found existing installation: lxml 6.1.0
    Uninstalling lxml-6.1.0:
      Successfully uninstalled lxml-6.1.0
  Attempting uninstall: charset_normalizer
    Found existing installation: charset-normalizer 3.4.7
    Uninstalling charset-normalizer-3.4.7:
      Successfully uninstalled charset-normalizer-3.4.7
ERROR: pip's dependency resolver does not currently take into accou

In [20]:
import json
from urllib.parse import urlparse

import trafilatura
from playwright.async_api import async_playwright


async def extract_url(url: str) -> dict:
    """
    Render URL bằng Playwright, sau đó dùng Trafilatura
    để extract main content + metadata.
    """

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page()

        try:
            await page.goto(
                url,
                wait_until="domcontentloaded",
                timeout=30_000,
            )

            # Cho JS động một khoảng ngắn để render nếu có
            try:
                await page.wait_for_load_state(
                    "networkidle",
                    timeout=10_000,
                )
            except Exception:
                # Một số site luôn có request nền nên networkidle không bao giờ xảy ra
                pass

            html = await page.content()

        finally:
            await browser.close()

    # Extract bằng Trafilatura
    result = trafilatura.extract(
        html,
        url=url,
        output_format="json",
        with_metadata=True,
        include_comments=False,
        include_tables=True,
        favor_precision=True,
    )

    if result is None:
        return {
            "url": url,
            "domain": urlparse(url).netloc,
            "success": False,
            "error": "Trafilatura could not extract content",
        }

    data = json.loads(result)

    # Chuẩn hóa output
    return {
        "url": data.get("url") or url,
        "domain": urlparse(url).netloc,
        "source": data.get("sitename") or data.get("hostname"),
        "title": data.get("title"),
        "author": data.get("author"),
        "published_date": data.get("date"),
        "description": data.get("description"),
        "text": data.get("text"),
        "success": True,
    }

In [23]:
import pandas as pd
r2ai_lucraftmeds = pd.DataFrame(columns=["doc_id","url","domain","source","title","author","published_date","description","text","success"])

In [24]:
r2ai_lucraftmeds

,doc_id,url,domain,source,title,author,published_date,description,text,success


In [17]:
urls = []
for i in range(len(ds["train"])):
    urls.append(ds["train"][i])

In [18]:
len(urls)

4394718

In [22]:
urls[0]

{'id': 583, 'url': 'http://special.vietnamplus.vn/benh_vien_nhan_ai'}

In [ ]:
rows = []

for item in urls:
    doc_id = item["id"]
    url = item["url"]

    article = await extract_url(url)

    if "error" in article:
        print("Error at:", url)
        print(article["error"])
        break

    article["doc_id"] = doc_id
    rows.append(article)

r2ai_lucraftmeds = pd.DataFrame(rows)